# Porto taxi trip-duration exploration

This notebook explores the Porto taxi dataset and establishes initial baselines. The target is the time from a passenger pickup to their destination. Only pickup, destination and request time are allowed as model inputs; completed GPS trajectories are used only to derive the target and assess data quality.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from cabrynt_trip_duration.preprocessing import (
    MAX_DURATION_MINUTES,
    MAX_SEGMENT_SPEED_KMH,
    duration_minutes,
    haversine_km,
    parse_polyline,
    preprocessing_reason,
    trajectory_statistics,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / 'data' / 'uci' / 'train.csv.zip'
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Training data not found at {TRAIN_PATH}')

## Load a representative sample

The full archive is intentionally not loaded in a notebook. The separate audit script scans it in chunks; this sample keeps exploration quick.

In [2]:
SAMPLE_ROWS = 10_000
raw_trips = pd.read_csv(TRAIN_PATH, nrows=SAMPLE_ROWS)

print(f'Loaded {len(raw_trips):,} rows')
display(raw_trips.head())
raw_trips.dtypes.to_frame('type')

Loaded 10,000 rows


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."
3,1372636854620000520,C,NaN,NaN,20000520,1372636854,A,False,"[[-8.574678,41.151951],[-8.574705,41.151942],[..."
4,1372637091620000337,C,NaN,NaN,20000337,1372637091,A,False,"[[-8.645994,41.18049],[-8.645949,41.180517],[-..."


,type
TRIP_ID,int64
CALL_TYPE,str
ORIGIN_CALL,float64
ORIGIN_STAND,float64
TAXI_ID,int64
TIMESTAMP,int64
DAY_TYPE,str
MISSING_DATA,bool
POLYLINE,str


## Parse trajectories and derive duration

GPS positions are sampled every 15 seconds. A trajectory with `n` points has `n - 1` time intervals, so duration is `(n - 1) * 15 seconds`.

In [3]:
trips = raw_trips.copy()
trips['points'] = trips['POLYLINE'].apply(parse_polyline)
trips['point_count'] = trips['points'].apply(
    lambda points: len(points) if points is not None else 0
)
trips['duration_minutes'] = trips['points'].apply(
    lambda points: duration_minutes(points) if points is not None else np.nan
)

example = trips.loc[trips['points'].notna()].iloc[0]
pd.Series({
    'point_count': example['point_count'],
    'start': example['points'][0],
    'destination': example['points'][-1],
    'duration_minutes': example['duration_minutes'],
})

point_count                             23
start               (-8.618643, 41.141412)
destination         (-8.630838, 41.154489)
duration_minutes                       5.5
dtype: object

## Apply the initial preprocessing policy

Exclude malformed or empty trajectories, fewer than two points, coordinates outside the Porto area, durations above four hours, and GPS jumps faster than 150 km/h. Repeated coordinates, `MISSING_DATA`, and short return trips remain in the data because they do not prove that the trip duration is invalid.

In [4]:
valid_points = trips['points'].notna()
statistics = trips.loc[valid_points, 'points'].apply(
    lambda points: pd.Series(trajectory_statistics(points))
)
trips.loc[valid_points, statistics.columns] = statistics
trips['preprocessing_reason'] = trips['points'].apply(preprocessing_reason)

quality_summary = trips['preprocessing_reason'].fillna('kept').value_counts()
quality_summary.loc['marked_missing_data_diagnostic'] = trips['MISSING_DATA'].sum()
quality_summary.to_frame('rows')

,rows
preprocessing_reason,
kept,8236
gps_speed_teleport,1595
too_few_points,122
empty_polyline,40
outside_porto_area,7
marked_missing_data_diagnostic,0


In [5]:
threshold_comparison = pd.Series({
    f'above_{limit}_kmh': (trips['max_segment_speed_kmh'] > limit).sum()
    for limit in [120, MAX_SEGMENT_SPEED_KMH, 200]
})
threshold_comparison.to_frame('flagged_trips')

,flagged_trips
above_120_kmh,2151
above_150.0_kmh,1601
above_200_kmh,770


## Prepare quote-time features

Observed GPS distance is retained below only for error analysis. It is not included in the feature list because it would not exist when a passenger requests a quote.

In [6]:
model_data = trips.loc[trips['preprocessing_reason'].isna()].copy()

model_data['pickup_longitude'] = model_data['points'].apply(lambda points: points[0][0])
model_data['pickup_latitude'] = model_data['points'].apply(lambda points: points[0][1])
model_data['destination_longitude'] = model_data['points'].apply(lambda points: points[-1][0])
model_data['destination_latitude'] = model_data['points'].apply(lambda points: points[-1][1])
model_data['straight_line_km'] = model_data.apply(
    lambda row: haversine_km(
        row['pickup_longitude'], row['pickup_latitude'],
        row['destination_longitude'], row['destination_latitude'],
    ),
    axis=1,
)
model_data['started_at'] = pd.to_datetime(model_data['TIMESTAMP'], unit='s', utc=True)
model_data['hour'] = model_data['started_at'].dt.hour
model_data['weekday'] = model_data['started_at'].dt.weekday
model_data['is_weekend'] = (model_data['weekday'] >= 5).astype(int)

print(f'Kept {len(model_data):,} of {len(trips):,} sampled rows')
model_data[['duration_minutes', 'straight_line_km', 'observed_distance_km']].describe()

Kept 8,236 of 10,000 sampled rows


,duration_minutes,straight_line_km,observed_distance_km
count,8236.000000,8236.000000,8236.000000
mean,11.399891,3.237432,5.217594
std,8.175851,2.707987,4.801724
min,0.250000,0.000000,0.000000
25%,6.750000,1.493110,2.342285
50%,10.000000,2.518022,3.801821
75%,14.000000,4.030621,6.375192
max,230.250000,42.246346,124.842078


## Use a chronological split

Training data must come before validation data in time. The final 15% stays unused until later experiments are stable.

In [7]:
model_data = model_data.sort_values('TIMESTAMP').reset_index(drop=True)
train_end = int(len(model_data) * 0.70)
validation_end = int(len(model_data) * 0.85)

train_data = model_data.iloc[:train_end]
validation_data = model_data.iloc[train_end:validation_end]
test_data = model_data.iloc[validation_end:]

pd.Series({
    'train': len(train_data),
    'validation': len(validation_data),
    'test_unused': len(test_data),
}).to_frame('rows')

,rows
train,5765
validation,1235
test_unused,1236


## Establish initial baselines

The median and fixed-speed estimates establish simple reference points. Gradient boosting is an initial tabular ML model, not a final claim of model quality.

In [8]:
def metrics(actual, predicted):
    return {
        'mae_minutes': mean_absolute_error(actual, predicted),
        'rmse_minutes': root_mean_squared_error(actual, predicted),
    }

target = 'duration_minutes'
features = [
    'pickup_longitude', 'pickup_latitude',
    'destination_longitude', 'destination_latitude',
    'straight_line_km', 'hour', 'weekday', 'is_weekend',
]

median_predictions = np.full(len(validation_data), train_data[target].median())
fixed_speed_predictions = validation_data['straight_line_km'] / 30 * 60

model = HistGradientBoostingRegressor(
    learning_rate=0.05, max_iter=200, random_state=42
)
model.fit(train_data[features], train_data[target])
model_predictions = model.predict(validation_data[features])

pd.DataFrame(
    [
        metrics(validation_data[target], median_predictions),
        metrics(validation_data[target], fixed_speed_predictions),
        metrics(validation_data[target], model_predictions),
    ],
    index=['training_median', 'fixed_speed', 'gradient_boosting'],
).round(2)

,mae_minutes,rmse_minutes
training_median,5.12,8.72
fixed_speed,6.43,9.96
gradient_boosting,3.98,7.65


## Inspect validation errors

The largest errors show where a later routing baseline or additional quote-time features may help.

In [9]:
validation_errors = validation_data[[
    'TRIP_ID', 'duration_minutes', 'straight_line_km',
    'observed_distance_km', 'max_segment_speed_kmh', 'hour',
]].copy()
validation_errors['predicted_minutes'] = model_predictions
validation_errors['signed_error_minutes'] = (
    validation_errors['predicted_minutes'] - validation_errors['duration_minutes']
)
validation_errors['absolute_error_minutes'] = validation_errors['signed_error_minutes'].abs()

validation_errors.sort_values('absolute_error_minutes', ascending=False).head(20)

,TRIP_ID,duration_minutes,straight_line_km,observed_distance_km,max_segment_speed_kmh,hour,predicted_minutes,signed_error_minutes,absolute_error_minutes
6439,1372780111620000114,125.50,5.307207,45.534712,119.869355,15,16.539823,-108.960177,108.960177
6472,1372780506620000492,101.75,0.627264,5.839837,60.542389,15,10.544294,-91.205706,91.205706
6759,1372784936620000279,92.00,4.102502,13.795342,55.247332,17,14.760778,-77.239222,77.239222
6117,1372774690620000226,78.50,6.099553,30.083683,109.056287,14,16.255407,-62.244593,62.244593
6126,1372774829620000246,58.00,0.530020,11.846681,128.509698,14,13.945769,-44.054231,44.054231
6688,1372783784620000319,52.00,2.015524,11.206621,83.767141,16,11.641390,-40.358610,40.358610
6974,1372789039620000224,54.50,8.825983,18.497663,102.413291,18,19.139471,-35.360529,35.360529
6259,1372776864620000030,44.25,0.241084,14.763122,71.781706,14,10.965220,-33.284780,33.284780
5924,1372772162620000562,57.00,12.006158,20.047145,115.258079,13,25.313856,-31.686144,31.686144
6135,1372774960620000007,37.00,0.330477,8.057613,63.318746,14,6.049352,-30.950648,30.950648
